# 08 (Bonus) - Browser Navigation with MCP

**This one's optional.** Everything else in this course only needs Python. This notebook also needs [Node.js](https://nodejs.org) installed (for `npx`), and what you build here is **notebook-only** -- it doesn't plug into the deployed Telegram bot without extra infrastructure (more on that at the end). Treat this as a stretch goal, not core material.

## What you'll build

An agent that can actually **navigate real websites** -- click things, fill forms, read live pages -- using [Playwright MCP](https://github.com/microsoft/playwright-mcp), Microsoft's official browser-automation server, connected via **MCP** (Model Context Protocol): a standard way for agents to talk to external tool servers.


In [ ]:
%pip install -q openai-agents

import shutil

from agents import set_default_openai_key

# Don't share or commit this notebook with your key filled in.
OPENAI_API_KEY = "sk-..."  # <-- paste your key here
set_default_openai_key(OPENAI_API_KEY)

if shutil.which("npx") is None:
    print("npx not found -- install Node.js (https://nodejs.org) before continuing.")
else:
    print("Ready to go.")

## Connecting to Playwright MCP

`MCPServerStdio` spawns a local process and talks to it directly -- in this case, `npx @playwright/mcp@latest`, which launches a real (headless) browser and exposes it as a set of tools over MCP. We connect once with `.connect()` so it stays alive across the next few cells (rather than `async with`, which would only stay open for one cell), and clean up at the end of the notebook.

The first run downloads the package and a browser, so it may take a moment.


In [ ]:
from agents.mcp import MCPServerStdio

playwright_server = MCPServerStdio(
    params={"command": "npx", "args": ["-y", "@playwright/mcp@latest", "--headless"]},
    client_session_timeout_seconds=60,
)
await playwright_server.connect()

tools = await playwright_server.list_tools()
print(f"Connected -- {len(tools)} tools available, e.g.:")
for t in tools[:6]:
    print(" -", t.name)

## Using it directly on an agent

Just like `tools=[...]` for custom tools, agents take `mcp_servers=[...]` for MCP servers -- every tool the server exposes becomes available to the agent.


In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Browser Agent",
    instructions="You browse real websites using your browser tools and report back what you find.",
    mcp_servers=[playwright_server],
)

result = await Runner.run(agent, "Go to example.com and tell me what the page says.")
print(result.final_output)

That worked, but notice the agent now has **all 24 raw browser tools** directly available (`browser_click`, `browser_navigate`, `browser_evaluate`, ...). That's a lot of low-level surface area to hand to an agent that also needs to do other things -- it's easy for a general-purpose assistant to get distracted by tools that don't apply to most questions.

## A cleaner pattern: wrap it as a tool

`agent.as_tool(...)` turns a whole agent into a single callable tool for *another* agent. The browser agent still has all 24 raw tools -- but whoever uses it as a tool only sees one thing: `browse_website(task)`. This is the same shape as every other tool in this course (one function, one job), it just happens to be backed by a full agent instead of a few lines of Python.


In [ ]:
browse_tool = agent.as_tool(
    tool_name="browse_website",
    tool_description="Browse a live website to answer a question or complete a task on it.",
)

main_agent = Agent(
    name="Main Assistant",
    instructions="Use browse_website when you need to look at a real, live webpage.",
    tools=[browse_tool],
)

print("Main agent's tools:", [t.name for t in main_agent.tools])

In [ ]:
result = await Runner.run(main_agent, "What's the page title at example.com?")
print(result.final_output)

`main_agent` never sees `browser_click` or any of the other 24 raw tools -- it just sees `browse_website`, calls it with a task description, and gets back a text answer. Internally, that call runs the *entire* browser agent (its own reasoning loop, its own tool calls) and hands back just the final result. This is exactly how you'd add browsing to an agent that already has several other tools, without cluttering its decision-making with browser internals.


## Cleaning up

Run this when you're done -- it stops the browser process `npx` started. Skipping this leaves an orphaned browser process running in the background.


In [ ]:
await playwright_server.cleanup()
print("Cleaned up.")

## Why this doesn't run in the Telegram bot (yet)

`MCPServerStdio` spawns `npx` as a real subprocess, which launches a real browser process. Your deployed bot runs on Vercel's **Python** Functions -- a separate runtime from Node.js, so `npx` isn't available there, and even if it were, running a full browser inside a serverless function isn't practical (large binaries, ephemeral filesystem, memory limits). This is the same wall `ComputerTool` hits (see notebook 04).

The fix, if you wanted this in the deployed bot, isn't a local process at all -- it's connecting to a browser/MCP server that's already running somewhere else, over the network, using `MCPServerSse` or `MCPServerStreamableHttp` instead of `MCPServerStdio`. That means standing up (or paying for) an always-on browser service, e.g. [Browserbase](https://www.browserbase.com/) -- real infrastructure beyond a one-file course project, and a good next step if you want to keep going after the course.
